# public

In [ ]:
#!/usr/bin/env python3
"""
Sample-level MiTRI read statistics with FASTA output
statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation, outputforfa.gzformat
"""

import os
import re
import logging
import gzip
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List, Tuple
import pandas as pd
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["BLCA", "LUNG", "OSCC", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
STATUSES = ["Normal", "Tumor"]
N_PROCESSES = 75

# Path configuration
BASE_MITRI = "/path/to/project/results_V3/cancers_V3.1"
BASE_CSV = "/path/to/project/data_V3/CSVs_20251208"
OUTPUT_BASE = "/path/to/project/results_V3/summary/MiTRI_reads_sample_level_fa"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def load_taxid_mapping(hierarchy_file: str) -> Dict[str, List[int]]:
    """
    Load microbe-name to tax_id mapping from V7_taxid_hierarchy.txt
    Returns:
    {microbe_name: [tax_id1, tax_id2, ...], ...}
    """
    microbe_to_taxids = defaultdict(list)
    try:
        df = pd.read_csv(hierarchy_file, sep='\t')
        # textrows, textspecies_scientific_nametotax_idmapping
        for _, row in df.iterrows():
            species_name = str(row['species_scientific_name'])
            tax_id = int(row['tax_id'])
            # textspeciestextintextandtextfortext
            microbe_name = re.sub(r'[\s/]+', '_', species_name)
            microbe_to_taxids[microbe_name].append(tax_id)
            print(f"OK Loaded taxid mapping: {len(microbe_to_taxids)} unique microbe names")
    except Exception as e:
        print(f"Failed Error loading taxid hierarchy: {e}")
        return dict(microbe_to_taxids)


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_mitri_stats")
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
        fh = logging.FileHandler(output_dir / f"{cancer}_mitri_stats_processing.log")
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_mitri_reads(txt_path: str) -> Dict[str, Dict[str, str]]:
    """
    Extract reads from MiTRI BAM.txt files(read_id, flag, sequence)
    Returns:
    reads_dict: {read_id: {'flag': flag, 'seq': sequence}, ...}
    """
    reads_dict = {}
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                flag = fields[1]
                sequence = fields[9]
                # Treat reads as valid when the sequence is not *
                if sequence != '*':
                    reads_dict[read_id] = {
                        'flag': flag,
                        'seq': sequence
                    }
    except FileNotFoundError:
        pass
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_dict


def aggregate_mitri_reads_with_info(cancer: str, status: str, sample: str,     microbes: List[str],     microbe_to_taxids: Dict[str, List[int]],
    logger) -> Tuple[Dict, Dict, Dict]:
"""
Aggregate sample MiTRI reads and record microbe source, taxid, flag, and sequence for each read
Returns:
read_info: {read_id: {'flag': flag, 'seq': seq}, ...}
read_to_microbes: {read_id: [microbe1, microbe2, ...], ...}
read_to_taxids: {read_id: [taxid1, taxid2, ...], ...}
"""
read_info = {} # textreadflagandsequence
read_to_microbes = defaultdict(list) # recordeachreadtextmicrobetext
read_to_taxids = defaultdict(set) # recordeachreadtextalltaxids
for microbe in microbes:
    mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /         microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
    if not mitri_txt.exists():
        continue
    # extracttextmicrobeMiTRI reads
    microbe_reads = extract_mitri_reads(str(mitri_txt))
    if len(microbe_reads) > 0:
        # textmicrobetaxids
        microbe_taxids = microbe_to_taxids.get(microbe, [])
        for read_id, read_data in microbe_reads.items():
            # textreadtextinformation(onlytext, textto)
            if read_id not in read_info:
                read_info[read_id] = read_data
                # recordmicrobe
                read_to_microbes[read_id].append(microbe)
                # textmicrobealltaxids
                for taxid in microbe_taxids:
                    read_to_taxids[read_id].add(taxid)
                    # texttaxidsforlisttextaftertextprocess
                    read_to_taxids = {k: list(v) for k, v in read_to_taxids.items()}
                    return read_info, dict(read_to_microbes), read_to_taxids


def save_sample_mitri_reads_as_fasta(cancer: str, status: str, sample: str,
    read_info: Dict,
    read_to_microbes: Dict,
    read_to_taxids: Dict,
    output_dir: Path) -> int:
"""
savesampleMiTRI readsforFASTAformat(text)
Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
Returns:
savereadscount
"""
# createOutput filespath
# createOutput filespath
output_fa_dir = output_dir / "fa"
output_fa_dir.mkdir(parents=True, exist_ok=True)
fasta_file = output_fa_dir / f"{cancer}.{status}.{sample}.MiTRI_reads.fa.gz"
if not read_info:
    return 0
    count = 0
    try:
        with gzip.open(fasta_file, 'wt') as f:
            for read_id in sorted(read_info.keys()):
                # textreadinformation
                flag = read_info[read_id]['flag']
                sequence = read_info[read_id]['seq']
                # textmicrobeandtaxids
                microbes = read_to_microbes.get(read_id, [])
                taxids = read_to_taxids.get(read_id, [])
                # buildannotationrows
                # microbestext
                microbes_str = ','.join(microbes)
                # taxidsformat: YP:Z:taxid1,taxid2,...
                taxids_str = f"YP:Z:{','.join(map(str, taxids))}" if taxids else "YP:Z:"
                # textFASTAannotationrows
                # >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
                header = f">{cancer}|{status}|{sample}|{read_id}|{flag}|{microbes_str}|{len(microbes)}|{len(taxids)}|{taxids_str}"
                # textFASTAformat
                f.write(f"{header}\n")
                f.write(f"{sequence}\n")
                count += 1
    except Exception as e:
        logging.error(f"Error writing FASTA file {fasta_file}: {e}")
        return 0
    return count


def process_sample(args):
    """
    processtextsamples: statisticsallMiTRI readstextsaveforFASTAformat
    """
    cancer, status, sample, microbes, microbe_to_taxids, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"Processing {cancer}/{status}/{sample}")
    # textMiTRI reads(textdetailedinformation)
    read_info, read_to_microbes, read_to_taxids = aggregate_mitri_reads_with_info(
        cancer, status, sample, microbes, microbe_to_taxids, logger
)
if len(read_info) == 0:
    logger.warning(f"{sample}: No MiTRI reads found")
    return None
    num_microbes_used = len(set([m for microbes_list in read_to_microbes.values()         for m in microbes_list]))
    logger.info(f"{sample}: Total MiTRI reads = {len(read_info)} from {num_microbes_used} microbes")
    # savesampleMiTRI readsforFASTAformat
    saved_count = save_sample_mitri_reads_as_fasta(
        cancer, status, sample, read_info,
        read_to_microbes, read_to_taxids, output_dir
)
if saved_count > 0:
    logger.info(f"Completed {cancer}/{status}/{sample}: saved {saved_count} reads to FASTA")
else:
    logger.error(f"Failed to save FASTA for {cancer}/{status}/{sample}")
    return None
    # returntextstatistics
    summary = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'total_mitri_reads': len(read_info),
        'num_microbes': num_microbes_used,
        'avg_microbes_per_read': sum(len(read_to_microbes[r]) for r in read_info) / len(read_info),
        'multi_match_reads': sum(1 for r in read_info if len(read_to_microbes[r]) > 1),
        'multi_match_pct': sum(1 for r in read_info if len(read_to_microbes[r]) > 1) / len(read_info) * 100
    }
    return summary


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext(fromMiTRIresultsdirectory)"""
    mitri_dir = Path(BASE_MITRI) / cancer / "03.coverage" / status
    if not mitri_dir.exists():
        return []
    # fromtextmicrobesdirectorytextsample columntext
    samples = set()
    for microbe_dir in mitri_dir.iterdir():
        if microbe_dir.is_dir():
            for txt_file in microbe_dir.glob("*_rna_output.pathseq_sorted.bam.txt"):
                sample = txt_file.name.replace("_rna_output.pathseq_sorted.bam.txt", "")
                samples.add(sample)
                return sorted(list(samples))


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"{cancer}_V4.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer(cancer: str, output_base: Path, microbe_to_taxids: Dict):
    """processtextcancer typeallsample"""
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== Starting processing for {cancer} ===")
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, microbe_to_taxids,                 output_base, f"{cancer}_mitri_stats"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_summaries = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for summary in pool.imap_unordered(process_sample, tasks):
                    if summary is not None:
                        all_summaries.append(summary)
                        # savecancer typetextstatistics
                        if all_summaries:
                            df_summary = pd.DataFrame(all_summaries)
                            summary_file = output_base / f"{cancer}_summary.tsv"
                            df_summary.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            logger.info(f"=== Completed {cancer}: {len(all_summaries)} samples ===")
                            return all_summaries


def create_visualizations(df_summary: pd.DataFrame, output_dir: Path):
    """createstatisticstext"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typeMiTRI readscount
        ax1 = axes[0, 0]
        cancer_stats = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average MiTRI Reads Count')
        ax1.set_title('Average MiTRI Reads by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRI readscounttext
        ax2 = axes[0, 1]
        df_summary['total_mitri_reads'].hist(bins=50, ax=ax2, color='green', edgecolor='black')
        ax2.set_xlabel('MiTRI Reads Count')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distribution of MiTRI Reads Count')
        ax2.axvline(df_summary['total_mitri_reads'].median(), color='red', linestyle='--',
            label=f"Median: {df_summary['total_mitri_reads'].median():.0f}")
        ax2.legend()
        # 3. Normal vs Tumorcompare
        ax3 = axes[0, 2]
        status_stats = df_summary.groupby('status')['total_mitri_reads'].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['lightblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Average MiTRI Reads')
        ax3.set_title('MiTRI Reads: Normal vs Tumor')
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. microbecounttext
        ax4 = axes[1, 0]
        df_summary['num_microbes'].hist(bins=30, ax=ax4, color='purple', edgecolor='black')
        ax4.set_xlabel('Number of Microbes')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Distribution of Microbes Count per Sample')
        ax4.axvline(df_summary['num_microbes'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['num_microbes'].mean():.1f}")
        ax4.legend()
        # 5. textmatchedreadstext
        ax5 = axes[1, 1]
        df_summary['multi_match_pct'].hist(bins=50, ax=ax5, color='orange', edgecolor='black')
        ax5.set_xlabel('Multi-Match Reads %')
        ax5.set_ylabel('Frequency')
        ax5.set_title('Distribution of Multi-Match Reads %')
        ax5.axvline(df_summary['multi_match_pct'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['multi_match_pct'].mean():.1f}%")
        ax5.legend()
        # 6. microbecount vs MiTRI reads
        ax6 = axes[1, 2]
        scatter = ax6.scatter(df_summary['num_microbes'], df_summary['total_mitri_reads'],
            c=df_summary['multi_match_pct'], cmap='viridis', alpha=0.6, s=30)
        ax6.set_xlabel('Number of Microbes')
        ax6.set_ylabel('Total MiTRI Reads')
        ax6.set_title('Microbes Count vs MiTRI Reads')
        plt.colorbar(scatter, ax=ax6, label='Multi-Match %')
        ax6.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'mitri_reads_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df_summary: pd.DataFrame, output_dir: Path):
    """generatedetailedstatistics report"""
    report_file = output_dir / "mitri_reads_statistics_report.txt"
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MiTRI Reads sampletextstatistics report (FASTAformatoutput)\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        # Overall statistics
        f.write("[1]Overall statistics\n")
        f.write("-"*80 + "\n")
        f.write(f"textsample count: {len(df_summary)}\n")
        f.write(f"total cancer types: {df_summary['cancer'].nunique()}\n")
        f.write(f"\naverage MiTRI reads per sample: {df_summary['total_mitri_reads'].mean():.0f}\n")
        f.write(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}\n")
        f.write(f"average microbes using each read: {df_summary['avg_microbes_per_read'].mean():.2f}\n")
        f.write(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%\n")
        f.write("\n")
        # Summarize by cancer type
        f.write("[2]statistics by cancer type\n")
        f.write("-"*80 + "\n")
        cancer_stats = df_summary.groupby('cancer').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        cancer_stats.columns = ['sample count', 'average reads', 'median reads', 'average microbes', 'averagetextmatched%']
        f.write(cancer_stats.to_string())
        f.write("\n\n")
        # textstatistics
        f.write("[3]Normal vs Tumor statistics\n")
        f.write("-"*80 + "\n")
        status_stats = df_summary.groupby('status').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        f.write(status_stats.to_string())
        f.write("\n\n")
        # TOPsample
        f.write("[4]Top 10 samples by MiTRI reads\n")
        f.write("-"*80 + "\n")
        top_samples = df_summary.nlargest(10, 'total_mitri_reads')[
            ['cancer', 'status', 'sample', 'total_mitri_reads', 'num_microbes']
        ]
        f.write(top_samples.to_string(index=False))
        f.write("\n\n")
        # outputformattext
        f.write("[5]Output file format notes\n")
        f.write("-"*80 + "\n")
        f.write("File format: FASTA (gzip compressed)\n")
        f.write("Filename: {cancer}.{status}.{sample}.MiTRI_reads.fa.gz\n")
        f.write("Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids\n")
        f.write(" - microbes_used: textmicrobe list\n")
        f.write(" - taxids: YP:Z:prefix followed by a comma-separated taxid list\n")
        f.write("\ntext:\n")
        f.write(">BRCA|Normal|SRR11600335|SRR11600335.35235387|16|Acinetobacter_johnsonii,Enterobacter_asburiae|2|2|YP:Z:61645,1217662\n")
        f.write("ATCGATCGATCG...\n")
        f.write("\n")
        f.write("="*80 + "\n")
        f.write("End of report\n")
        f.write("="*80 + "\n")
        print(f"OK Statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"MiTRI Reads sampletextstatisticstextoutputforFASTAformat")
    print(f"statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # loadtaxidmapping
    print("textloadmicrobetaxidmapping...")
    microbe_to_taxids = load_taxid_mapping(TAXID_HIERARCHY_FILE)
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_summaries = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_summaries = process_cancer(cancer, output_base, microbe_to_taxids)
        all_summaries.extend(cancer_summaries)
        # savetext
        if all_summaries:
            df_summary = pd.DataFrame(all_summaries)
            summary_file = output_base / "all_samples_summary.tsv"
            df_summary.to_csv(summary_file, sep='\t', index=False)
            print(f"\nOK Saved global summary: {summary_file}")
            # generatestatistics report
            print("\n" + "="*80)
            print("generatestatistics report...")
            print("="*80)
            generate_statistics_report(df_summary, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_summary)}")
            print(f"total cancer types: {df_summary['cancer'].nunique()}")
            print(f"\naveragetextsampleMiTRI reads: {df_summary['total_mitri_reads'].mean():.0f}")
            print(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}")
            print(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%")
            print(f"\n[textcancer typeaverageMiTRI reads]")
            cancer_avg = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
            for cancer, avg in cancer_avg.items():
                print(f" {cancer:6s}: {avg:8.0f}")
                print(f"\n[Normal vs Tumor]")
                status_avg = df_summary.groupby('status')['total_mitri_reads'].mean()
                for status, avg in status_avg.items():
                    print(f" {status:6s}: {avg:8.0f}")
                    # createtext
                    print("\n" + "="*80)
                    print("generatetext...")
                    print("="*80)
                    create_visualizations(df_summary, output_base)
                    elapsed_time = time.time() - start_time
                    print(f"\n{'='*80}")
                    print(f"MiTRI read statisticscompleted!")
                    print(f"output directory: {OUTPUT_BASE}")
                    print(f"text: {elapsed_time/60:.2f} text")
                    print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()

# inhouse

## R NR

In [ ]:
#!/usr/bin/env python3
"""
Sample-level MiTRI read statistics with FASTA output
statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation, outputforfa.gzformat
"""

import os
import re
import logging
import gzip
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List, Tuple
import pandas as pd
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["CRC"]
STATUSES = ["NR", "R"]
N_PROCESSES = 75

# Path configuration
BASE_MITRI = "/path/to/project/results_V3/special"
BASE_CSV = "/path/to/project/data/CSVs_20251114"
OUTPUT_BASE = "/path/to/project/results_V3/summary/MiTRI_reads_R_NR_sample_level_fa"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def load_taxid_mapping(hierarchy_file: str) -> Dict[str, List[int]]:
    """
    Load microbe-name to tax_id mapping from V7_taxid_hierarchy.txt
    Returns:
    {microbe_name: [tax_id1, tax_id2, ...], ...}
    """
    microbe_to_taxids = defaultdict(list)
    try:
        df = pd.read_csv(hierarchy_file, sep='\t')
        # textrows, textspecies_scientific_nametotax_idmapping
        for _, row in df.iterrows():
            species_name = str(row['species_scientific_name'])
            tax_id = int(row['tax_id'])
            # textspeciestextintextandtextfortext
            microbe_name = re.sub(r'[\s/]+', '_', species_name)
            microbe_to_taxids[microbe_name].append(tax_id)
            print(f"OK Loaded taxid mapping: {len(microbe_to_taxids)} unique microbe names")
    except Exception as e:
        print(f"Failed Error loading taxid hierarchy: {e}")
        return dict(microbe_to_taxids)


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_mitri_stats")
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
        fh = logging.FileHandler(output_dir / f"{cancer}_mitri_stats_processing.log")
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_mitri_reads(txt_path: str) -> Dict[str, Dict[str, str]]:
    """
    Extract reads from MiTRI BAM.txt files(read_id, flag, sequence)
    Returns:
    reads_dict: {read_id: {'flag': flag, 'seq': sequence}, ...}
    """
    reads_dict = {}
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                flag = fields[1]
                sequence = fields[9]
                # Treat reads as valid when the sequence is not *
                if sequence != '*':
                    reads_dict[read_id] = {
                        'flag': flag,
                        'seq': sequence
                    }
    except FileNotFoundError:
        pass
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_dict


def aggregate_mitri_reads_with_info(cancer: str, status: str, sample: str,     microbes: List[str],     microbe_to_taxids: Dict[str, List[int]],
    logger) -> Tuple[Dict, Dict, Dict]:
"""
Aggregate sample MiTRI reads and record microbe source, taxid, flag, and sequence for each read
Returns:
read_info: {read_id: {'flag': flag, 'seq': seq}, ...}
read_to_microbes: {read_id: [microbe1, microbe2, ...], ...}
read_to_taxids: {read_id: [taxid1, taxid2, ...], ...}
"""
read_info = {} # textreadflagandsequence
read_to_microbes = defaultdict(list) # recordeachreadtextmicrobetext
read_to_taxids = defaultdict(set) # recordeachreadtextalltaxids
for microbe in microbes:
    mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /         microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
    if not mitri_txt.exists():
        continue
    # extracttextmicrobeMiTRI reads
    microbe_reads = extract_mitri_reads(str(mitri_txt))
    if len(microbe_reads) > 0:
        # textmicrobetaxids
        microbe_taxids = microbe_to_taxids.get(microbe, [])
        for read_id, read_data in microbe_reads.items():
            # textreadtextinformation(onlytext, textto)
            if read_id not in read_info:
                read_info[read_id] = read_data
                # recordmicrobe
                read_to_microbes[read_id].append(microbe)
                # textmicrobealltaxids
                for taxid in microbe_taxids:
                    read_to_taxids[read_id].add(taxid)
                    # texttaxidsforlisttextaftertextprocess
                    read_to_taxids = {k: list(v) for k, v in read_to_taxids.items()}
                    return read_info, dict(read_to_microbes), read_to_taxids


def save_sample_mitri_reads_as_fasta(cancer: str, status: str, sample: str,
    read_info: Dict,
    read_to_microbes: Dict,
    read_to_taxids: Dict,
    output_dir: Path) -> int:
"""
savesampleMiTRI readsforFASTAformat(text)
Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
Returns:
savereadscount
"""
# createOutput filespath
output_fa_dir = output_dir / "fa"
output_fa_dir.mkdir(parents=True, exist_ok=True)
fasta_file = output_fa_dir / f"{cancer}.{status}.{sample}.MiTRI_reads.fa.gz"
if not read_info:
    return 0
    count = 0
    try:
        with gzip.open(fasta_file, 'wt') as f:
            for read_id in sorted(read_info.keys()):
                # textreadinformation
                flag = read_info[read_id]['flag']
                sequence = read_info[read_id]['seq']
                # textmicrobeandtaxids
                microbes = read_to_microbes.get(read_id, [])
                taxids = read_to_taxids.get(read_id, [])
                # buildannotationrows
                # microbestext
                microbes_str = ','.join(microbes)
                # taxidsformat: YP:Z:taxid1,taxid2,...
                taxids_str = f"YP:Z:{','.join(map(str, taxids))}" if taxids else "YP:Z:"
                # textFASTAannotationrows
                # >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
                header = f">{cancer}|{status}|{sample}|{read_id}|{flag}|{microbes_str}|{len(microbes)}|{len(taxids)}|{taxids_str}"
                # textFASTAformat
                f.write(f"{header}\n")
                f.write(f"{sequence}\n")
                count += 1
    except Exception as e:
        logging.error(f"Error writing FASTA file {fasta_file}: {e}")
        return 0
    return count


def process_sample(args):
    """
    processtextsamples: statisticsallMiTRI readstextsaveforFASTAformat
    """
    cancer, status, sample, microbes, microbe_to_taxids, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"Processing {cancer}/{status}/{sample}")
    # textMiTRI reads(textdetailedinformation)
    read_info, read_to_microbes, read_to_taxids = aggregate_mitri_reads_with_info(
        cancer, status, sample, microbes, microbe_to_taxids, logger
)
if len(read_info) == 0:
    logger.warning(f"{sample}: No MiTRI reads found")
    return None
    num_microbes_used = len(set([m for microbes_list in read_to_microbes.values()         for m in microbes_list]))
    logger.info(f"{sample}: Total MiTRI reads = {len(read_info)} from {num_microbes_used} microbes")
    # savesampleMiTRI readsforFASTAformat
    saved_count = save_sample_mitri_reads_as_fasta(
        cancer, status, sample, read_info,
        read_to_microbes, read_to_taxids, output_dir
)
if saved_count > 0:
    logger.info(f"Completed {cancer}/{status}/{sample}: saved {saved_count} reads to FASTA")
else:
    logger.error(f"Failed to save FASTA for {cancer}/{status}/{sample}")
    return None
    # returntextstatistics
    summary = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'total_mitri_reads': len(read_info),
        'num_microbes': num_microbes_used,
        'avg_microbes_per_read': sum(len(read_to_microbes[r]) for r in read_info) / len(read_info),
        'multi_match_reads': sum(1 for r in read_info if len(read_to_microbes[r]) > 1),
        'multi_match_pct': sum(1 for r in read_info if len(read_to_microbes[r]) > 1) / len(read_info) * 100
    }
    return summary


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext(fromMiTRIresultsdirectory)"""
    mitri_dir = Path(BASE_MITRI) / cancer / "03.coverage" / status
    if not mitri_dir.exists():
        return []
    # fromtextmicrobesdirectorytextsample columntext
    samples = set()
    for microbe_dir in mitri_dir.iterdir():
        if microbe_dir.is_dir():
            for txt_file in microbe_dir.glob("*_rna_output.pathseq_sorted.bam.txt"):
                sample = txt_file.name.replace("_rna_output.pathseq_sorted.bam.txt", "")
                samples.add(sample)
                return sorted(list(samples))


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"CRC_R_NR_paired.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer(cancer: str, output_base: Path, microbe_to_taxids: Dict):
    """processtextcancer typeallsample"""
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== Starting processing for {cancer} ===")
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, microbe_to_taxids,                 output_base, f"{cancer}_mitri_stats"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_summaries = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for summary in pool.imap_unordered(process_sample, tasks):
                    if summary is not None:
                        all_summaries.append(summary)
                        # savecancer typetextstatistics
                        if all_summaries:
                            df_summary = pd.DataFrame(all_summaries)
                            summary_file = output_base / f"{cancer}_summary.tsv"
                            df_summary.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            logger.info(f"=== Completed {cancer}: {len(all_summaries)} samples ===")
                            return all_summaries


def create_visualizations(df_summary: pd.DataFrame, output_dir: Path):
    """createstatisticstext"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typeMiTRI readscount
        ax1 = axes[0, 0]
        cancer_stats = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average MiTRI Reads Count')
        ax1.set_title('Average MiTRI Reads by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRI readscounttext
        ax2 = axes[0, 1]
        df_summary['total_mitri_reads'].hist(bins=50, ax=ax2, color='green', edgecolor='black')
        ax2.set_xlabel('MiTRI Reads Count')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distribution of MiTRI Reads Count')
        ax2.axvline(df_summary['total_mitri_reads'].median(), color='red', linestyle='--',
            label=f"Median: {df_summary['total_mitri_reads'].median():.0f}")
        ax2.legend()
        # 3. Normal vs Tumorcompare
        ax3 = axes[0, 2]
        status_stats = df_summary.groupby('status')['total_mitri_reads'].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['lightblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Average MiTRI Reads')
        ax3.set_title('MiTRI Reads: Normal vs Tumor')
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. microbecounttext
        ax4 = axes[1, 0]
        df_summary['num_microbes'].hist(bins=30, ax=ax4, color='purple', edgecolor='black')
        ax4.set_xlabel('Number of Microbes')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Distribution of Microbes Count per Sample')
        ax4.axvline(df_summary['num_microbes'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['num_microbes'].mean():.1f}")
        ax4.legend()
        # 5. textmatchedreadstext
        ax5 = axes[1, 1]
        df_summary['multi_match_pct'].hist(bins=50, ax=ax5, color='orange', edgecolor='black')
        ax5.set_xlabel('Multi-Match Reads %')
        ax5.set_ylabel('Frequency')
        ax5.set_title('Distribution of Multi-Match Reads %')
        ax5.axvline(df_summary['multi_match_pct'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['multi_match_pct'].mean():.1f}%")
        ax5.legend()
        # 6. microbecount vs MiTRI reads
        ax6 = axes[1, 2]
        scatter = ax6.scatter(df_summary['num_microbes'], df_summary['total_mitri_reads'],
            c=df_summary['multi_match_pct'], cmap='viridis', alpha=0.6, s=30)
        ax6.set_xlabel('Number of Microbes')
        ax6.set_ylabel('Total MiTRI Reads')
        ax6.set_title('Microbes Count vs MiTRI Reads')
        plt.colorbar(scatter, ax=ax6, label='Multi-Match %')
        ax6.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'mitri_reads_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df_summary: pd.DataFrame, output_dir: Path):
    """generatedetailedstatistics report"""
    report_file = output_dir / "mitri_reads_statistics_report.txt"
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MiTRI Reads sampletextstatistics report (FASTAformatoutput)\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        # Overall statistics
        f.write("[1]Overall statistics\n")
        f.write("-"*80 + "\n")
        f.write(f"textsample count: {len(df_summary)}\n")
        f.write(f"total cancer types: {df_summary['cancer'].nunique()}\n")
        f.write(f"\naverage MiTRI reads per sample: {df_summary['total_mitri_reads'].mean():.0f}\n")
        f.write(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}\n")
        f.write(f"average microbes using each read: {df_summary['avg_microbes_per_read'].mean():.2f}\n")
        f.write(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%\n")
        f.write("\n")
        # Summarize by cancer type
        f.write("[2]statistics by cancer type\n")
        f.write("-"*80 + "\n")
        cancer_stats = df_summary.groupby('cancer').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        cancer_stats.columns = ['sample count', 'average reads', 'median reads', 'average microbes', 'averagetextmatched%']
        f.write(cancer_stats.to_string())
        f.write("\n\n")
        # textstatistics
        f.write("[3]Normal vs Tumor statistics\n")
        f.write("-"*80 + "\n")
        status_stats = df_summary.groupby('status').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        f.write(status_stats.to_string())
        f.write("\n\n")
        # TOPsample
        f.write("[4]Top 10 samples by MiTRI reads\n")
        f.write("-"*80 + "\n")
        top_samples = df_summary.nlargest(10, 'total_mitri_reads')[
            ['cancer', 'status', 'sample', 'total_mitri_reads', 'num_microbes']
        ]
        f.write(top_samples.to_string(index=False))
        f.write("\n\n")
        # outputformattext
        f.write("[5]Output file format notes\n")
        f.write("-"*80 + "\n")
        f.write("File format: FASTA (gzip compressed)\n")
        f.write("Filename: {cancer}.{status}.{sample}.MiTRI_reads.fa.gz\n")
        f.write("Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids\n")
        f.write(" - microbes_used: textmicrobe list\n")
        f.write(" - taxids: YP:Z:prefix followed by a comma-separated taxid list\n")
        f.write("\ntext:\n")
        f.write(">BRCA|Normal|SRR11600335|SRR11600335.35235387|16|Acinetobacter_johnsonii,Enterobacter_asburiae|2|2|YP:Z:61645,1217662\n")
        f.write("ATCGATCGATCG...\n")
        f.write("\n")
        f.write("="*80 + "\n")
        f.write("End of report\n")
        f.write("="*80 + "\n")
        print(f"OK Statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"MiTRI Reads sampletextstatisticstextoutputforFASTAformat")
    print(f"statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # loadtaxidmapping
    print("textloadmicrobetaxidmapping...")
    microbe_to_taxids = load_taxid_mapping(TAXID_HIERARCHY_FILE)
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_summaries = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_summaries = process_cancer(cancer, output_base, microbe_to_taxids)
        all_summaries.extend(cancer_summaries)
        # savetext
        if all_summaries:
            df_summary = pd.DataFrame(all_summaries)
            summary_file = output_base / "all_samples_summary.tsv"
            df_summary.to_csv(summary_file, sep='\t', index=False)
            print(f"\nOK Saved global summary: {summary_file}")
            # generatestatistics report
            print("\n" + "="*80)
            print("generatestatistics report...")
            print("="*80)
            generate_statistics_report(df_summary, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_summary)}")
            print(f"total cancer types: {df_summary['cancer'].nunique()}")
            print(f"\naveragetextsampleMiTRI reads: {df_summary['total_mitri_reads'].mean():.0f}")
            print(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}")
            print(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%")
            print(f"\n[textcancer typeaverageMiTRI reads]")
            cancer_avg = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
            for cancer, avg in cancer_avg.items():
                print(f" {cancer:6s}: {avg:8.0f}")
                print(f"\n[Normal vs Tumor]")
                status_avg = df_summary.groupby('status')['total_mitri_reads'].mean()
                for status, avg in status_avg.items():
                    print(f" {status:6s}: {avg:8.0f}")
                    # createtext
                    print("\n" + "="*80)
                    print("generatetext...")
                    print("="*80)
                    create_visualizations(df_summary, output_base)
                    elapsed_time = time.time() - start_time
                    print(f"\n{'='*80}")
                    print(f"MiTRI read statisticscompleted!")
                    print(f"output directory: {OUTPUT_BASE}")
                    print(f"text: {elapsed_time/60:.2f} text")
                    print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()

# MS

In [ ]:
#!/usr/bin/env python3
"""
Sample-level MiTRI read statistics with FASTA output
statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation, outputforfa.gzformat
"""

import os
import re
import logging
import gzip
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List, Tuple
import pandas as pd
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["CRC_MS"]
STATUSES = ["T"]
N_PROCESSES = 75

# Path configuration
BASE_MITRI = "/path/to/project/results_V3/special"
BASE_CSV = "/path/to/project/data_V3/CSVs_20251208"
OUTPUT_BASE = "/path/to/project/results_V3/summary/MiTRI_reads_MS_T_sample_level_fa_V3"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def load_taxid_mapping(hierarchy_file: str) -> Dict[str, List[int]]:
    """
    Load microbe-name to tax_id mapping from V7_taxid_hierarchy.txt
    Returns:
    {microbe_name: [tax_id1, tax_id2, ...], ...}
    """
    microbe_to_taxids = defaultdict(list)
    try:
        df = pd.read_csv(hierarchy_file, sep='\t')
        # textrows, textspecies_scientific_nametotax_idmapping
        for _, row in df.iterrows():
            species_name = str(row['species_scientific_name'])
            tax_id = int(row['tax_id'])
            # textspeciestextintextandtextfortext
            microbe_name = re.sub(r'[\s/]+', '_', species_name)
            microbe_to_taxids[microbe_name].append(tax_id)
            print(f"OK Loaded taxid mapping: {len(microbe_to_taxids)} unique microbe names")
    except Exception as e:
        print(f"Failed Error loading taxid hierarchy: {e}")
        return dict(microbe_to_taxids)


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_mitri_stats")
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
        fh = logging.FileHandler(output_dir / f"{cancer}_mitri_stats_processing.log")
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_mitri_reads(txt_path: str) -> Dict[str, Dict[str, str]]:
    """
    Extract reads from MiTRI BAM.txt files(read_id, flag, sequence)
    Returns:
    reads_dict: {read_id: {'flag': flag, 'seq': sequence}, ...}
    """
    reads_dict = {}
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                flag = fields[1]
                sequence = fields[9]
                # Treat reads as valid when the sequence is not *
                if sequence != '*':
                    reads_dict[read_id] = {
                        'flag': flag,
                        'seq': sequence
                    }
    except FileNotFoundError:
        pass
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_dict


def aggregate_mitri_reads_with_info(cancer: str, status: str, sample: str,     microbes: List[str],     microbe_to_taxids: Dict[str, List[int]],
    logger) -> Tuple[Dict, Dict, Dict]:
"""
Aggregate sample MiTRI reads and record microbe source, taxid, flag, and sequence for each read
Returns:
read_info: {read_id: {'flag': flag, 'seq': seq}, ...}
read_to_microbes: {read_id: [microbe1, microbe2, ...], ...}
read_to_taxids: {read_id: [taxid1, taxid2, ...], ...}
"""
read_info = {} # textreadflagandsequence
read_to_microbes = defaultdict(list) # recordeachreadtextmicrobetext
read_to_taxids = defaultdict(set) # recordeachreadtextalltaxids
for microbe in microbes:
    mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /         microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
    if not mitri_txt.exists():
        continue
    # extracttextmicrobeMiTRI reads
    microbe_reads = extract_mitri_reads(str(mitri_txt))
    if len(microbe_reads) > 0:
        # textmicrobetaxids
        microbe_taxids = microbe_to_taxids.get(microbe, [])
        for read_id, read_data in microbe_reads.items():
            # textreadtextinformation(onlytext, textto)
            if read_id not in read_info:
                read_info[read_id] = read_data
                # recordmicrobe
                read_to_microbes[read_id].append(microbe)
                # textmicrobealltaxids
                for taxid in microbe_taxids:
                    read_to_taxids[read_id].add(taxid)
                    # texttaxidsforlisttextaftertextprocess
                    read_to_taxids = {k: list(v) for k, v in read_to_taxids.items()}
                    return read_info, dict(read_to_microbes), read_to_taxids


def save_sample_mitri_reads_as_fasta(cancer: str, status: str, sample: str,
    read_info: Dict,
    read_to_microbes: Dict,
    read_to_taxids: Dict,
    output_dir: Path) -> int:
"""
savesampleMiTRI readsforFASTAformat(text)
Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
Returns:
savereadscount
"""
# createOutput filespath
output_fa_dir = output_dir / "fa"
output_fa_dir.mkdir(parents=True, exist_ok=True)
fasta_file = output_fa_dir / f"{cancer}.{status}.{sample}.MiTRI_reads.fa.gz"
if not read_info:
    return 0
    count = 0
    try:
        with gzip.open(fasta_file, 'wt') as f:
            for read_id in sorted(read_info.keys()):
                # textreadinformation
                flag = read_info[read_id]['flag']
                sequence = read_info[read_id]['seq']
                # textmicrobeandtaxids
                microbes = read_to_microbes.get(read_id, [])
                taxids = read_to_taxids.get(read_id, [])
                # buildannotationrows
                # microbestext
                microbes_str = ','.join(microbes)
                # taxidsformat: YP:Z:taxid1,taxid2,...
                taxids_str = f"YP:Z:{','.join(map(str, taxids))}" if taxids else "YP:Z:"
                # textFASTAannotationrows
                # >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
                header = f">{cancer}|{status}|{sample}|{read_id}|{flag}|{microbes_str}|{len(microbes)}|{len(taxids)}|{taxids_str}"
                # textFASTAformat
                f.write(f"{header}\n")
                f.write(f"{sequence}\n")
                count += 1
    except Exception as e:
        logging.error(f"Error writing FASTA file {fasta_file}: {e}")
        return 0
    return count


def process_sample(args):
    """
    processtextsamples: statisticsallMiTRI readstextsaveforFASTAformat
    """
    cancer, status, sample, microbes, microbe_to_taxids, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"Processing {cancer}/{status}/{sample}")
    # textMiTRI reads(textdetailedinformation)
    read_info, read_to_microbes, read_to_taxids = aggregate_mitri_reads_with_info(
        cancer, status, sample, microbes, microbe_to_taxids, logger
)
if len(read_info) == 0:
    logger.warning(f"{sample}: No MiTRI reads found")
    return None
    num_microbes_used = len(set([m for microbes_list in read_to_microbes.values()         for m in microbes_list]))
    logger.info(f"{sample}: Total MiTRI reads = {len(read_info)} from {num_microbes_used} microbes")
    # savesampleMiTRI readsforFASTAformat
    saved_count = save_sample_mitri_reads_as_fasta(
        cancer, status, sample, read_info,
        read_to_microbes, read_to_taxids, output_dir
)
if saved_count > 0:
    logger.info(f"Completed {cancer}/{status}/{sample}: saved {saved_count} reads to FASTA")
else:
    logger.error(f"Failed to save FASTA for {cancer}/{status}/{sample}")
    return None
    # returntextstatistics
    summary = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'total_mitri_reads': len(read_info),
        'num_microbes': num_microbes_used,
        'avg_microbes_per_read': sum(len(read_to_microbes[r]) for r in read_info) / len(read_info),
        'multi_match_reads': sum(1 for r in read_info if len(read_to_microbes[r]) > 1),
        'multi_match_pct': sum(1 for r in read_info if len(read_to_microbes[r]) > 1) / len(read_info) * 100
    }
    return summary


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext(fromMiTRIresultsdirectory)"""
    mitri_dir = Path(BASE_MITRI) / cancer / "03.coverage" / status
    if not mitri_dir.exists():
        return []
    # fromtextmicrobesdirectorytextsample columntext
    samples = set()
    for microbe_dir in mitri_dir.iterdir():
        if microbe_dir.is_dir():
            for txt_file in microbe_dir.glob("*_rna_output.pathseq_sorted.bam.txt"):
                sample = txt_file.name.replace("_rna_output.pathseq_sorted.bam.txt", "")
                samples.add(sample)
                return sorted(list(samples))


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"CRC_MS_V5.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer(cancer: str, output_base: Path, microbe_to_taxids: Dict):
    """processtextcancer typeallsample"""
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== Starting processing for {cancer} ===")
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, microbe_to_taxids,                 output_base, f"{cancer}_mitri_stats"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_summaries = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for summary in pool.imap_unordered(process_sample, tasks):
                    if summary is not None:
                        all_summaries.append(summary)
                        # savecancer typetextstatistics
                        if all_summaries:
                            df_summary = pd.DataFrame(all_summaries)
                            summary_file = output_base / f"{cancer}_summary.tsv"
                            df_summary.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            logger.info(f"=== Completed {cancer}: {len(all_summaries)} samples ===")
                            return all_summaries


def create_visualizations(df_summary: pd.DataFrame, output_dir: Path):
    """createstatisticstext"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typeMiTRI readscount
        ax1 = axes[0, 0]
        cancer_stats = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average MiTRI Reads Count')
        ax1.set_title('Average MiTRI Reads by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRI readscounttext
        ax2 = axes[0, 1]
        df_summary['total_mitri_reads'].hist(bins=50, ax=ax2, color='green', edgecolor='black')
        ax2.set_xlabel('MiTRI Reads Count')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distribution of MiTRI Reads Count')
        ax2.axvline(df_summary['total_mitri_reads'].median(), color='red', linestyle='--',
            label=f"Median: {df_summary['total_mitri_reads'].median():.0f}")
        ax2.legend()
        # 3. Normal vs Tumorcompare
        ax3 = axes[0, 2]
        status_stats = df_summary.groupby('status')['total_mitri_reads'].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['lightblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Average MiTRI Reads')
        ax3.set_title('MiTRI Reads: Normal vs Tumor')
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. microbecounttext
        ax4 = axes[1, 0]
        df_summary['num_microbes'].hist(bins=30, ax=ax4, color='purple', edgecolor='black')
        ax4.set_xlabel('Number of Microbes')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Distribution of Microbes Count per Sample')
        ax4.axvline(df_summary['num_microbes'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['num_microbes'].mean():.1f}")
        ax4.legend()
        # 5. textmatchedreadstext
        ax5 = axes[1, 1]
        df_summary['multi_match_pct'].hist(bins=50, ax=ax5, color='orange', edgecolor='black')
        ax5.set_xlabel('Multi-Match Reads %')
        ax5.set_ylabel('Frequency')
        ax5.set_title('Distribution of Multi-Match Reads %')
        ax5.axvline(df_summary['multi_match_pct'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['multi_match_pct'].mean():.1f}%")
        ax5.legend()
        # 6. microbecount vs MiTRI reads
        ax6 = axes[1, 2]
        scatter = ax6.scatter(df_summary['num_microbes'], df_summary['total_mitri_reads'],
            c=df_summary['multi_match_pct'], cmap='viridis', alpha=0.6, s=30)
        ax6.set_xlabel('Number of Microbes')
        ax6.set_ylabel('Total MiTRI Reads')
        ax6.set_title('Microbes Count vs MiTRI Reads')
        plt.colorbar(scatter, ax=ax6, label='Multi-Match %')
        ax6.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'mitri_reads_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df_summary: pd.DataFrame, output_dir: Path):
    """generatedetailedstatistics report"""
    report_file = output_dir / "mitri_reads_statistics_report.txt"
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MiTRI Reads sampletextstatistics report (FASTAformatoutput)\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        # Overall statistics
        f.write("[1]Overall statistics\n")
        f.write("-"*80 + "\n")
        f.write(f"textsample count: {len(df_summary)}\n")
        f.write(f"total cancer types: {df_summary['cancer'].nunique()}\n")
        f.write(f"\naverage MiTRI reads per sample: {df_summary['total_mitri_reads'].mean():.0f}\n")
        f.write(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}\n")
        f.write(f"average microbes using each read: {df_summary['avg_microbes_per_read'].mean():.2f}\n")
        f.write(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%\n")
        f.write("\n")
        # Summarize by cancer type
        f.write("[2]statistics by cancer type\n")
        f.write("-"*80 + "\n")
        cancer_stats = df_summary.groupby('cancer').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        cancer_stats.columns = ['sample count', 'average reads', 'median reads', 'average microbes', 'averagetextmatched%']
        f.write(cancer_stats.to_string())
        f.write("\n\n")
        # textstatistics
        f.write("[3]Normal vs Tumor statistics\n")
        f.write("-"*80 + "\n")
        status_stats = df_summary.groupby('status').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        f.write(status_stats.to_string())
        f.write("\n\n")
        # TOPsample
        f.write("[4]Top 10 samples by MiTRI reads\n")
        f.write("-"*80 + "\n")
        top_samples = df_summary.nlargest(10, 'total_mitri_reads')[
            ['cancer', 'status', 'sample', 'total_mitri_reads', 'num_microbes']
        ]
        f.write(top_samples.to_string(index=False))
        f.write("\n\n")
        # outputformattext
        f.write("[5]Output file format notes\n")
        f.write("-"*80 + "\n")
        f.write("File format: FASTA (gzip compressed)\n")
        f.write("Filename: {cancer}.{status}.{sample}.MiTRI_reads.fa.gz\n")
        f.write("Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids\n")
        f.write(" - microbes_used: textmicrobe list\n")
        f.write(" - taxids: YP:Z:prefix followed by a comma-separated taxid list\n")
        f.write("\ntext:\n")
        f.write(">BRCA|Normal|SRR11600335|SRR11600335.35235387|16|Acinetobacter_johnsonii,Enterobacter_asburiae|2|2|YP:Z:61645,1217662\n")
        f.write("ATCGATCGATCG...\n")
        f.write("\n")
        f.write("="*80 + "\n")
        f.write("End of report\n")
        f.write("="*80 + "\n")
        print(f"OK Statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"MiTRI Reads sampletextstatisticstextoutputforFASTAformat")
    print(f"statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # loadtaxidmapping
    print("textloadmicrobetaxidmapping...")
    microbe_to_taxids = load_taxid_mapping(TAXID_HIERARCHY_FILE)
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_summaries = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_summaries = process_cancer(cancer, output_base, microbe_to_taxids)
        all_summaries.extend(cancer_summaries)
        # savetext
        if all_summaries:
            df_summary = pd.DataFrame(all_summaries)
            summary_file = output_base / "all_samples_summary.tsv"
            df_summary.to_csv(summary_file, sep='\t', index=False)
            print(f"\nOK Saved global summary: {summary_file}")
            # generatestatistics report
            print("\n" + "="*80)
            print("generatestatistics report...")
            print("="*80)
            generate_statistics_report(df_summary, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_summary)}")
            print(f"total cancer types: {df_summary['cancer'].nunique()}")
            print(f"\naveragetextsampleMiTRI reads: {df_summary['total_mitri_reads'].mean():.0f}")
            print(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}")
            print(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%")
            print(f"\n[textcancer typeaverageMiTRI reads]")
            cancer_avg = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
            for cancer, avg in cancer_avg.items():
                print(f" {cancer:6s}: {avg:8.0f}")
                print(f"\n[Normal vs Tumor]")
                status_avg = df_summary.groupby('status')['total_mitri_reads'].mean()
                for status, avg in status_avg.items():
                    print(f" {status:6s}: {avg:8.0f}")
                    # createtext
                    print("\n" + "="*80)
                    print("generatetext...")
                    print("="*80)
                    create_visualizations(df_summary, output_base)
                    elapsed_time = time.time() - start_time
                    print(f"\n{'='*80}")
                    print(f"MiTRI read statisticscompleted!")
                    print(f"output directory: {OUTPUT_BASE}")
                    print(f"text: {elapsed_time/60:.2f} text")
                    print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()

# bassfish

In [ ]:
#!/usr/bin/env python3
"""
Sample-level MiTRI read statistics with FASTA output
statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation, outputforfa.gzformat
"""

import os
import re
import logging
import gzip
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List, Tuple
import pandas as pd
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["CRC_bassfish"]
STATUSES = ["T"]
N_PROCESSES = 75

# Path configuration
BASE_MITRI = "/path/to/project/results_V3/special"
BASE_CSV = "/path/to/project/data_V3/CSVs_20251208"
OUTPUT_BASE = "/path/to/project/results_V3/summary/MiTRI_reads_bassfish_T_sample_level_fa_V3"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def load_taxid_mapping(hierarchy_file: str) -> Dict[str, List[int]]:
    """
    Load microbe-name to tax_id mapping from V7_taxid_hierarchy.txt
    Returns:
    {microbe_name: [tax_id1, tax_id2, ...], ...}
    """
    microbe_to_taxids = defaultdict(list)
    try:
        df = pd.read_csv(hierarchy_file, sep='\t')
        # textrows, textspecies_scientific_nametotax_idmapping
        for _, row in df.iterrows():
            species_name = str(row['species_scientific_name'])
            tax_id = int(row['tax_id'])
            # textspeciestextintextandtextfortext
            microbe_name = re.sub(r'[\s/]+', '_', species_name)
            microbe_to_taxids[microbe_name].append(tax_id)
            print(f"OK Loaded taxid mapping: {len(microbe_to_taxids)} unique microbe names")
    except Exception as e:
        print(f"Failed Error loading taxid hierarchy: {e}")
        return dict(microbe_to_taxids)


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_mitri_stats")
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
        fh = logging.FileHandler(output_dir / f"{cancer}_mitri_stats_processing.log")
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_mitri_reads(txt_path: str) -> Dict[str, Dict[str, str]]:
    """
    Extract reads from MiTRI BAM.txt files(read_id, flag, sequence)
    Returns:
    reads_dict: {read_id: {'flag': flag, 'seq': sequence}, ...}
    """
    reads_dict = {}
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                flag = fields[1]
                sequence = fields[9]
                # Treat reads as valid when the sequence is not *
                if sequence != '*':
                    reads_dict[read_id] = {
                        'flag': flag,
                        'seq': sequence
                    }
    except FileNotFoundError:
        pass
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_dict


def aggregate_mitri_reads_with_info(cancer: str, status: str, sample: str,     microbes: List[str],     microbe_to_taxids: Dict[str, List[int]],
    logger) -> Tuple[Dict, Dict, Dict]:
"""
Aggregate sample MiTRI reads and record microbe source, taxid, flag, and sequence for each read
Returns:
read_info: {read_id: {'flag': flag, 'seq': seq}, ...}
read_to_microbes: {read_id: [microbe1, microbe2, ...], ...}
read_to_taxids: {read_id: [taxid1, taxid2, ...], ...}
"""
read_info = {} # textreadflagandsequence
read_to_microbes = defaultdict(list) # recordeachreadtextmicrobetext
read_to_taxids = defaultdict(set) # recordeachreadtextalltaxids
for microbe in microbes:
    mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /         microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
    if not mitri_txt.exists():
        continue
    # extracttextmicrobeMiTRI reads
    microbe_reads = extract_mitri_reads(str(mitri_txt))
    if len(microbe_reads) > 0:
        # textmicrobetaxids
        microbe_taxids = microbe_to_taxids.get(microbe, [])
        for read_id, read_data in microbe_reads.items():
            # textreadtextinformation(onlytext, textto)
            if read_id not in read_info:
                read_info[read_id] = read_data
                # recordmicrobe
                read_to_microbes[read_id].append(microbe)
                # textmicrobealltaxids
                for taxid in microbe_taxids:
                    read_to_taxids[read_id].add(taxid)
                    # texttaxidsforlisttextaftertextprocess
                    read_to_taxids = {k: list(v) for k, v in read_to_taxids.items()}
                    return read_info, dict(read_to_microbes), read_to_taxids


def save_sample_mitri_reads_as_fasta(cancer: str, status: str, sample: str,
    read_info: Dict,
    read_to_microbes: Dict,
    read_to_taxids: Dict,
    output_dir: Path) -> int:
"""
savesampleMiTRI readsforFASTAformat(text)
Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
Returns:
savereadscount
"""
# createOutput filespath
output_fa_dir = output_dir / "fa"
output_fa_dir.mkdir(parents=True, exist_ok=True)
fasta_file = output_fa_dir / f"{cancer}.{status}.{sample}.MiTRI_reads.fa.gz"
if not read_info:
    return 0
    count = 0
    try:
        with gzip.open(fasta_file, 'wt') as f:
            for read_id in sorted(read_info.keys()):
                # textreadinformation
                flag = read_info[read_id]['flag']
                sequence = read_info[read_id]['seq']
                # textmicrobeandtaxids
                microbes = read_to_microbes.get(read_id, [])
                taxids = read_to_taxids.get(read_id, [])
                # buildannotationrows
                # microbestext
                microbes_str = ','.join(microbes)
                # taxidsformat: YP:Z:taxid1,taxid2,...
                taxids_str = f"YP:Z:{','.join(map(str, taxids))}" if taxids else "YP:Z:"
                # textFASTAannotationrows
                # >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
                header = f">{cancer}|{status}|{sample}|{read_id}|{flag}|{microbes_str}|{len(microbes)}|{len(taxids)}|{taxids_str}"
                # textFASTAformat
                f.write(f"{header}\n")
                f.write(f"{sequence}\n")
                count += 1
    except Exception as e:
        logging.error(f"Error writing FASTA file {fasta_file}: {e}")
        return 0
    return count


def process_sample(args):
    """
    processtextsamples: statisticsallMiTRI readstextsaveforFASTAformat
    """
    cancer, status, sample, microbes, microbe_to_taxids, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"Processing {cancer}/{status}/{sample}")
    # textMiTRI reads(textdetailedinformation)
    read_info, read_to_microbes, read_to_taxids = aggregate_mitri_reads_with_info(
        cancer, status, sample, microbes, microbe_to_taxids, logger
)
if len(read_info) == 0:
    logger.warning(f"{sample}: No MiTRI reads found")
    return None
    num_microbes_used = len(set([m for microbes_list in read_to_microbes.values()         for m in microbes_list]))
    logger.info(f"{sample}: Total MiTRI reads = {len(read_info)} from {num_microbes_used} microbes")
    # savesampleMiTRI readsforFASTAformat
    saved_count = save_sample_mitri_reads_as_fasta(
        cancer, status, sample, read_info,
        read_to_microbes, read_to_taxids, output_dir
)
if saved_count > 0:
    logger.info(f"Completed {cancer}/{status}/{sample}: saved {saved_count} reads to FASTA")
else:
    logger.error(f"Failed to save FASTA for {cancer}/{status}/{sample}")
    return None
    # returntextstatistics
    summary = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'total_mitri_reads': len(read_info),
        'num_microbes': num_microbes_used,
        'avg_microbes_per_read': sum(len(read_to_microbes[r]) for r in read_info) / len(read_info),
        'multi_match_reads': sum(1 for r in read_info if len(read_to_microbes[r]) > 1),
        'multi_match_pct': sum(1 for r in read_info if len(read_to_microbes[r]) > 1) / len(read_info) * 100
    }
    return summary


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext(fromMiTRIresultsdirectory)"""
    mitri_dir = Path(BASE_MITRI) / cancer / "03.coverage" / status
    if not mitri_dir.exists():
        return []
    # fromtextmicrobesdirectorytextsample columntext
    samples = set()
    for microbe_dir in mitri_dir.iterdir():
        if microbe_dir.is_dir():
            for txt_file in microbe_dir.glob("*_rna_output.pathseq_sorted.bam.txt"):
                sample = txt_file.name.replace("_rna_output.pathseq_sorted.bam.txt", "")
                samples.add(sample)
                return sorted(list(samples))


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"CRC_bassfish.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer(cancer: str, output_base: Path, microbe_to_taxids: Dict):
    """processtextcancer typeallsample"""
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== Starting processing for {cancer} ===")
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, microbe_to_taxids,                 output_base, f"{cancer}_mitri_stats"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_summaries = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for summary in pool.imap_unordered(process_sample, tasks):
                    if summary is not None:
                        all_summaries.append(summary)
                        # savecancer typetextstatistics
                        if all_summaries:
                            df_summary = pd.DataFrame(all_summaries)
                            summary_file = output_base / f"{cancer}_summary.tsv"
                            df_summary.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            logger.info(f"=== Completed {cancer}: {len(all_summaries)} samples ===")
                            return all_summaries


def create_visualizations(df_summary: pd.DataFrame, output_dir: Path):
    """createstatisticstext"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typeMiTRI readscount
        ax1 = axes[0, 0]
        cancer_stats = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average MiTRI Reads Count')
        ax1.set_title('Average MiTRI Reads by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRI readscounttext
        ax2 = axes[0, 1]
        df_summary['total_mitri_reads'].hist(bins=50, ax=ax2, color='green', edgecolor='black')
        ax2.set_xlabel('MiTRI Reads Count')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distribution of MiTRI Reads Count')
        ax2.axvline(df_summary['total_mitri_reads'].median(), color='red', linestyle='--',
            label=f"Median: {df_summary['total_mitri_reads'].median():.0f}")
        ax2.legend()
        # 3. Normal vs Tumorcompare
        ax3 = axes[0, 2]
        status_stats = df_summary.groupby('status')['total_mitri_reads'].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['lightblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Average MiTRI Reads')
        ax3.set_title('MiTRI Reads: Normal vs Tumor')
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. microbecounttext
        ax4 = axes[1, 0]
        df_summary['num_microbes'].hist(bins=30, ax=ax4, color='purple', edgecolor='black')
        ax4.set_xlabel('Number of Microbes')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Distribution of Microbes Count per Sample')
        ax4.axvline(df_summary['num_microbes'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['num_microbes'].mean():.1f}")
        ax4.legend()
        # 5. textmatchedreadstext
        ax5 = axes[1, 1]
        df_summary['multi_match_pct'].hist(bins=50, ax=ax5, color='orange', edgecolor='black')
        ax5.set_xlabel('Multi-Match Reads %')
        ax5.set_ylabel('Frequency')
        ax5.set_title('Distribution of Multi-Match Reads %')
        ax5.axvline(df_summary['multi_match_pct'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['multi_match_pct'].mean():.1f}%")
        ax5.legend()
        # 6. microbecount vs MiTRI reads
        ax6 = axes[1, 2]
        scatter = ax6.scatter(df_summary['num_microbes'], df_summary['total_mitri_reads'],
            c=df_summary['multi_match_pct'], cmap='viridis', alpha=0.6, s=30)
        ax6.set_xlabel('Number of Microbes')
        ax6.set_ylabel('Total MiTRI Reads')
        ax6.set_title('Microbes Count vs MiTRI Reads')
        plt.colorbar(scatter, ax=ax6, label='Multi-Match %')
        ax6.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'mitri_reads_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df_summary: pd.DataFrame, output_dir: Path):
    """generatedetailedstatistics report"""
    report_file = output_dir / "mitri_reads_statistics_report.txt"
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MiTRI Reads sampletextstatistics report (FASTAformatoutput)\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        # Overall statistics
        f.write("[1]Overall statistics\n")
        f.write("-"*80 + "\n")
        f.write(f"textsample count: {len(df_summary)}\n")
        f.write(f"total cancer types: {df_summary['cancer'].nunique()}\n")
        f.write(f"\naverage MiTRI reads per sample: {df_summary['total_mitri_reads'].mean():.0f}\n")
        f.write(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}\n")
        f.write(f"average microbes using each read: {df_summary['avg_microbes_per_read'].mean():.2f}\n")
        f.write(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%\n")
        f.write("\n")
        # Summarize by cancer type
        f.write("[2]statistics by cancer type\n")
        f.write("-"*80 + "\n")
        cancer_stats = df_summary.groupby('cancer').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        cancer_stats.columns = ['sample count', 'average reads', 'median reads', 'average microbes', 'averagetextmatched%']
        f.write(cancer_stats.to_string())
        f.write("\n\n")
        # textstatistics
        f.write("[3]Normal vs Tumor statistics\n")
        f.write("-"*80 + "\n")
        status_stats = df_summary.groupby('status').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        f.write(status_stats.to_string())
        f.write("\n\n")
        # TOPsample
        f.write("[4]Top 10 samples by MiTRI reads\n")
        f.write("-"*80 + "\n")
        top_samples = df_summary.nlargest(10, 'total_mitri_reads')[
            ['cancer', 'status', 'sample', 'total_mitri_reads', 'num_microbes']
        ]
        f.write(top_samples.to_string(index=False))
        f.write("\n\n")
        # outputformattext
        f.write("[5]Output file format notes\n")
        f.write("-"*80 + "\n")
        f.write("File format: FASTA (gzip compressed)\n")
        f.write("Filename: {cancer}.{status}.{sample}.MiTRI_reads.fa.gz\n")
        f.write("Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids\n")
        f.write(" - microbes_used: textmicrobe list\n")
        f.write(" - taxids: YP:Z:prefix followed by a comma-separated taxid list\n")
        f.write("\ntext:\n")
        f.write(">BRCA|Normal|SRR11600335|SRR11600335.35235387|16|Acinetobacter_johnsonii,Enterobacter_asburiae|2|2|YP:Z:61645,1217662\n")
        f.write("ATCGATCGATCG...\n")
        f.write("\n")
        f.write("="*80 + "\n")
        f.write("End of report\n")
        f.write("="*80 + "\n")
        print(f"OK Statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"MiTRI Reads sampletextstatisticstextoutputforFASTAformat")
    print(f"statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # loadtaxidmapping
    print("textloadmicrobetaxidmapping...")
    microbe_to_taxids = load_taxid_mapping(TAXID_HIERARCHY_FILE)
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_summaries = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_summaries = process_cancer(cancer, output_base, microbe_to_taxids)
        all_summaries.extend(cancer_summaries)
        # savetext
        if all_summaries:
            df_summary = pd.DataFrame(all_summaries)
            summary_file = output_base / "all_samples_summary.tsv"
            df_summary.to_csv(summary_file, sep='\t', index=False)
            print(f"\nOK Saved global summary: {summary_file}")
            # generatestatistics report
            print("\n" + "="*80)
            print("generatestatistics report...")
            print("="*80)
            generate_statistics_report(df_summary, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_summary)}")
            print(f"total cancer types: {df_summary['cancer'].nunique()}")
            print(f"\naveragetextsampleMiTRI reads: {df_summary['total_mitri_reads'].mean():.0f}")
            print(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}")
            print(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%")
            print(f"\n[textcancer typeaverageMiTRI reads]")
            cancer_avg = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
            for cancer, avg in cancer_avg.items():
                print(f" {cancer:6s}: {avg:8.0f}")
                print(f"\n[Normal vs Tumor]")
                status_avg = df_summary.groupby('status')['total_mitri_reads'].mean()
                for status, avg in status_avg.items():
                    print(f" {status:6s}: {avg:8.0f}")
                    # createtext
                    print("\n" + "="*80)
                    print("generatetext...")
                    print("="*80)
                    create_visualizations(df_summary, output_base)
                    elapsed_time = time.time() - start_time
                    print(f"\n{'='*80}")
                    print(f"MiTRI read statisticscompleted!")
                    print(f"output directory: {OUTPUT_BASE}")
                    print(f"text: {elapsed_time/60:.2f} text")
                    print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()

# PDO

## text

In [ ]:
#!/usr/bin/env python3
"""
Sample-level MiTRI read statistics with FASTA output
statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation, outputforfa.gzformat
"""

import os
import re
import logging
import gzip
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List, Tuple
import pandas as pd
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["CRC_PDO"]
STATUSES = ["T"]
N_PROCESSES = 75

# Path configuration
BASE_MITRI = "/path/to/project/results_V3/special"
BASE_CSV = "/path/to/project/data/CSVs_20251114"
OUTPUT_BASE = "/path/to/project/results_V3/summary/MiTRI_reads_PDO_T_sample_level_fa_V3"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def load_taxid_mapping(hierarchy_file: str) -> Dict[str, List[int]]:
    """
    Load microbe-name to tax_id mapping from V7_taxid_hierarchy.txt
    Returns:
    {microbe_name: [tax_id1, tax_id2, ...], ...}
    """
    microbe_to_taxids = defaultdict(list)
    try:
        df = pd.read_csv(hierarchy_file, sep='\t')
        # textrows, textspecies_scientific_nametotax_idmapping
        for _, row in df.iterrows():
            species_name = str(row['species_scientific_name'])
            tax_id = int(row['tax_id'])
            # textspeciestextintextandtextfortext
            microbe_name = re.sub(r'[\s/]+', '_', species_name)
            microbe_to_taxids[microbe_name].append(tax_id)
            print(f"OK Loaded taxid mapping: {len(microbe_to_taxids)} unique microbe names")
    except Exception as e:
        print(f"Failed Error loading taxid hierarchy: {e}")
        return dict(microbe_to_taxids)


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_mitri_stats")
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
        fh = logging.FileHandler(output_dir / f"{cancer}_mitri_stats_processing.log")
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_mitri_reads(txt_path: str) -> Dict[str, Dict[str, str]]:
    """
    Extract reads from MiTRI BAM.txt files(read_id, flag, sequence)
    Returns:
    reads_dict: {read_id: {'flag': flag, 'seq': sequence}, ...}
    """
    reads_dict = {}
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                flag = fields[1]
                sequence = fields[9]
                # Treat reads as valid when the sequence is not *
                if sequence != '*':
                    reads_dict[read_id] = {
                        'flag': flag,
                        'seq': sequence
                    }
    except FileNotFoundError:
        pass
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_dict


def aggregate_mitri_reads_with_info(cancer: str, status: str, sample: str,     microbes: List[str],     microbe_to_taxids: Dict[str, List[int]],
    logger) -> Tuple[Dict, Dict, Dict]:
"""
Aggregate sample MiTRI reads and record microbe source, taxid, flag, and sequence for each read
Returns:
read_info: {read_id: {'flag': flag, 'seq': seq}, ...}
read_to_microbes: {read_id: [microbe1, microbe2, ...], ...}
read_to_taxids: {read_id: [taxid1, taxid2, ...], ...}
"""
read_info = {} # textreadflagandsequence
read_to_microbes = defaultdict(list) # recordeachreadtextmicrobetext
read_to_taxids = defaultdict(set) # recordeachreadtextalltaxids
for microbe in microbes:
    mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /         microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
    if not mitri_txt.exists():
        continue
    # extracttextmicrobeMiTRI reads
    microbe_reads = extract_mitri_reads(str(mitri_txt))
    if len(microbe_reads) > 0:
        # textmicrobetaxids
        microbe_taxids = microbe_to_taxids.get(microbe, [])
        for read_id, read_data in microbe_reads.items():
            # textreadtextinformation(onlytext, textto)
            if read_id not in read_info:
                read_info[read_id] = read_data
                # recordmicrobe
                read_to_microbes[read_id].append(microbe)
                # textmicrobealltaxids
                for taxid in microbe_taxids:
                    read_to_taxids[read_id].add(taxid)
                    # texttaxidsforlisttextaftertextprocess
                    read_to_taxids = {k: list(v) for k, v in read_to_taxids.items()}
                    return read_info, dict(read_to_microbes), read_to_taxids


def save_sample_mitri_reads_as_fasta(cancer: str, status: str, sample: str,
    read_info: Dict,
    read_to_microbes: Dict,
    read_to_taxids: Dict,
    output_dir: Path) -> int:
"""
savesampleMiTRI readsforFASTAformat(text)
Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
Returns:
savereadscount
"""
# createOutput filespath
output_fa_dir = output_dir / "fa"
output_fa_dir.mkdir(parents=True, exist_ok=True)
fasta_file = output_fa_dir / f"{cancer}.{status}.{sample}.MiTRI_reads.fa.gz"
if not read_info:
    return 0
    count = 0
    try:
        with gzip.open(fasta_file, 'wt') as f:
            for read_id in sorted(read_info.keys()):
                # textreadinformation
                flag = read_info[read_id]['flag']
                sequence = read_info[read_id]['seq']
                # textmicrobeandtaxids
                microbes = read_to_microbes.get(read_id, [])
                taxids = read_to_taxids.get(read_id, [])
                # buildannotationrows
                # microbestext
                microbes_str = ','.join(microbes)
                # taxidsformat: YP:Z:taxid1,taxid2,...
                taxids_str = f"YP:Z:{','.join(map(str, taxids))}" if taxids else "YP:Z:"
                # textFASTAannotationrows
                # >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
                header = f">{cancer}|{status}|{sample}|{read_id}|{flag}|{microbes_str}|{len(microbes)}|{len(taxids)}|{taxids_str}"
                # textFASTAformat
                f.write(f"{header}\n")
                f.write(f"{sequence}\n")
                count += 1
    except Exception as e:
        logging.error(f"Error writing FASTA file {fasta_file}: {e}")
        return 0
    return count


def process_sample(args):
    """
    processtextsamples: statisticsallMiTRI readstextsaveforFASTAformat
    """
    cancer, status, sample, microbes, microbe_to_taxids, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"Processing {cancer}/{status}/{sample}")
    # textMiTRI reads(textdetailedinformation)
    read_info, read_to_microbes, read_to_taxids = aggregate_mitri_reads_with_info(
        cancer, status, sample, microbes, microbe_to_taxids, logger
)
if len(read_info) == 0:
    logger.warning(f"{sample}: No MiTRI reads found")
    return None
    num_microbes_used = len(set([m for microbes_list in read_to_microbes.values()         for m in microbes_list]))
    logger.info(f"{sample}: Total MiTRI reads = {len(read_info)} from {num_microbes_used} microbes")
    # savesampleMiTRI readsforFASTAformat
    saved_count = save_sample_mitri_reads_as_fasta(
        cancer, status, sample, read_info,
        read_to_microbes, read_to_taxids, output_dir
)
if saved_count > 0:
    logger.info(f"Completed {cancer}/{status}/{sample}: saved {saved_count} reads to FASTA")
else:
    logger.error(f"Failed to save FASTA for {cancer}/{status}/{sample}")
    return None
    # returntextstatistics
    summary = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'total_mitri_reads': len(read_info),
        'num_microbes': num_microbes_used,
        'avg_microbes_per_read': sum(len(read_to_microbes[r]) for r in read_info) / len(read_info),
        'multi_match_reads': sum(1 for r in read_info if len(read_to_microbes[r]) > 1),
        'multi_match_pct': sum(1 for r in read_info if len(read_to_microbes[r]) > 1) / len(read_info) * 100
    }
    return summary


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext(fromMiTRIresultsdirectory)"""
    mitri_dir = Path(BASE_MITRI) / cancer / "03.coverage" / status
    if not mitri_dir.exists():
        return []
    # fromtextmicrobesdirectorytextsample columntext
    samples = set()
    for microbe_dir in mitri_dir.iterdir():
        if microbe_dir.is_dir():
            for txt_file in microbe_dir.glob("*_rna_output.pathseq_sorted.bam.txt"):
                sample = txt_file.name.replace("_rna_output.pathseq_sorted.bam.txt", "")
                samples.add(sample)
                return sorted(list(samples))
True

def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"CRC_bassfish.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer(cancer: str, output_base: Path, microbe_to_taxids: Dict):
    """processtextcancer typeallsample"""
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== Starting processing for {cancer} ===")
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, microbe_to_taxids,                 output_base, f"{cancer}_mitri_stats"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_summaries = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for summary in pool.imap_unordered(process_sample, tasks):
                    if summary is not None:
                        all_summaries.append(summary)
                        # savecancer typetextstatistics
                        if all_summaries:
                            df_summary = pd.DataFrame(all_summaries)
                            summary_file = output_base / f"{cancer}_summary.tsv"
                            df_summary.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            logger.info(f"=== Completed {cancer}: {len(all_summaries)} samples ===")
                            return all_summaries


def create_visualizations(df_summary: pd.DataFrame, output_dir: Path):
    """createstatisticstext"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typeMiTRI readscount
        ax1 = axes[0, 0]
        cancer_stats = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average MiTRI Reads Count')
        ax1.set_title('Average MiTRI Reads by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRI readscounttext
        ax2 = axes[0, 1]
        df_summary['total_mitri_reads'].hist(bins=50, ax=ax2, color='green', edgecolor='black')
        ax2.set_xlabel('MiTRI Reads Count')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distribution of MiTRI Reads Count')
        ax2.axvline(df_summary['total_mitri_reads'].median(), color='red', linestyle='--',
            label=f"Median: {df_summary['total_mitri_reads'].median():.0f}")
        ax2.legend()
        # 3. Normal vs Tumorcompare
        ax3 = axes[0, 2]
        status_stats = df_summary.groupby('status')['total_mitri_reads'].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['lightblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Average MiTRI Reads')
        ax3.set_title('MiTRI Reads: Normal vs Tumor')
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. microbecounttext
        ax4 = axes[1, 0]
        df_summary['num_microbes'].hist(bins=30, ax=ax4, color='purple', edgecolor='black')
        ax4.set_xlabel('Number of Microbes')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Distribution of Microbes Count per Sample')
        ax4.axvline(df_summary['num_microbes'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['num_microbes'].mean():.1f}")
        ax4.legend()
        # 5. textmatchedreadstext
        ax5 = axes[1, 1]
        df_summary['multi_match_pct'].hist(bins=50, ax=ax5, color='orange', edgecolor='black')
        ax5.set_xlabel('Multi-Match Reads %')
        ax5.set_ylabel('Frequency')
        ax5.set_title('Distribution of Multi-Match Reads %')
        ax5.axvline(df_summary['multi_match_pct'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['multi_match_pct'].mean():.1f}%")
        ax5.legend()
        # 6. microbecount vs MiTRI reads
        ax6 = axes[1, 2]
        scatter = ax6.scatter(df_summary['num_microbes'], df_summary['total_mitri_reads'],
            c=df_summary['multi_match_pct'], cmap='viridis', alpha=0.6, s=30)
        ax6.set_xlabel('Number of Microbes')
        ax6.set_ylabel('Total MiTRI Reads')
        ax6.set_title('Microbes Count vs MiTRI Reads')
        plt.colorbar(scatter, ax=ax6, label='Multi-Match %')
        ax6.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'mitri_reads_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df_summary: pd.DataFrame, output_dir: Path):
    """generatedetailedstatistics report"""
    report_file = output_dir / "mitri_reads_statistics_report.txt"
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MiTRI Reads sampletextstatistics report (FASTAformatoutput)\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        # Overall statistics
        f.write("[1]Overall statistics\n")
        f.write("-"*80 + "\n")
        f.write(f"textsample count: {len(df_summary)}\n")
        f.write(f"total cancer types: {df_summary['cancer'].nunique()}\n")
        f.write(f"\naverage MiTRI reads per sample: {df_summary['total_mitri_reads'].mean():.0f}\n")
        f.write(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}\n")
        f.write(f"average microbes using each read: {df_summary['avg_microbes_per_read'].mean():.2f}\n")
        f.write(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%\n")
        f.write("\n")
        # Summarize by cancer type
        f.write("[2]statistics by cancer type\n")
        f.write("-"*80 + "\n")
        cancer_stats = df_summary.groupby('cancer').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        cancer_stats.columns = ['sample count', 'average reads', 'median reads', 'average microbes', 'averagetextmatched%']
        f.write(cancer_stats.to_string())
        f.write("\n\n")
        # textstatistics
        f.write("[3]Normal vs Tumor statistics\n")
        f.write("-"*80 + "\n")
        status_stats = df_summary.groupby('status').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        f.write(status_stats.to_string())
        f.write("\n\n")
        # TOPsample
        f.write("[4]Top 10 samples by MiTRI reads\n")
        f.write("-"*80 + "\n")
        top_samples = df_summary.nlargest(10, 'total_mitri_reads')[
            ['cancer', 'status', 'sample', 'total_mitri_reads', 'num_microbes']
        ]
        f.write(top_samples.to_string(index=False))
        f.write("\n\n")
        # outputformattext
        f.write("[5]Output file format notes\n")
        f.write("-"*80 + "\n")
        f.write("File format: FASTA (gzip compressed)\n")
        f.write("Filename: {cancer}.{status}.{sample}.MiTRI_reads.fa.gz\n")
        f.write("Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids\n")
        f.write(" - microbes_used: textmicrobe list\n")
        f.write(" - taxids: YP:Z:prefix followed by a comma-separated taxid list\n")
        f.write("\ntext:\n")
        f.write(">BRCA|Normal|SRR11600335|SRR11600335.35235387|16|Acinetobacter_johnsonii,Enterobacter_asburiae|2|2|YP:Z:61645,1217662\n")
        f.write("ATCGATCGATCG...\n")
        f.write("\n")
        f.write("="*80 + "\n")
        f.write("End of report\n")
        f.write("="*80 + "\n")
        print(f"OK Statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"MiTRI Reads sampletextstatisticstextoutputforFASTAformat")
    print(f"statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # loadtaxidmapping
    print("textloadmicrobetaxidmapping...")
    microbe_to_taxids = load_taxid_mapping(TAXID_HIERARCHY_FILE)
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_summaries = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_summaries = process_cancer(cancer, output_base, microbe_to_taxids)
        all_summaries.extend(cancer_summaries)
        # savetext
        if all_summaries:
            df_summary = pd.DataFrame(all_summaries)
            summary_file = output_base / "all_samples_summary.tsv"
            df_summary.to_csv(summary_file, sep='\t', index=False)
            print(f"\nOK Saved global summary: {summary_file}")
            # generatestatistics report
            print("\n" + "="*80)
            print("generatestatistics report...")
            print("="*80)
            generate_statistics_report(df_summary, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_summary)}")
            print(f"total cancer types: {df_summary['cancer'].nunique()}")
            print(f"\naveragetextsampleMiTRI reads: {df_summary['total_mitri_reads'].mean():.0f}")
            print(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}")
            print(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%")
            print(f"\n[textcancer typeaverageMiTRI reads]")
            cancer_avg = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
            for cancer, avg in cancer_avg.items():
                print(f" {cancer:6s}: {avg:8.0f}")
                print(f"\n[Normal vs Tumor]")
                status_avg = df_summary.groupby('status')['total_mitri_reads'].mean()
                for status, avg in status_avg.items():
                    print(f" {status:6s}: {avg:8.0f}")
                    # createtext
                    print("\n" + "="*80)
                    print("generatetext...")
                    print("="*80)
                    create_visualizations(df_summary, output_base)
                    elapsed_time = time.time() - start_time
                    print(f"\n{'='*80}")
                    print(f"MiTRI read statisticscompleted!")
                    print(f"output directory: {OUTPUT_BASE}")
                    print(f"text: {elapsed_time/60:.2f} text")
                    print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()